# NLP Fundamentals & Tokenization  
## Interactive Jupyter Notebook

Welcome to the NLP and Tokenization lesson.

In previous lessons, we worked with:

```text
Recommendation data → ratings and numerical features
Computer Vision → pixels and brightness values
```

Now we move into text.

Text is unstructured data. A computer cannot directly understand words like:

```text
I drink tea
```

Before a model can process text, we must convert it into numbers.

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why text must be converted into numbers.
2. Build a small vocabulary manually.
3. Convert raw text into token IDs.
4. Handle unknown words using `[UNK]`.
5. Apply padding using `[PAD]`.
6. Compare word-level, character-level, and sub-word tokenization.
7. Explain what a context window is.
8. Understand why token IDs are not meaning.
9. Use basic token counting ideas.
10. Understand safe API key storage concepts.

# Part 1: Why Text Must Become Numbers

Computers are calculators.

They can do math with numbers, but text has no natural numerical value.

For example:

```text
"I drink tea"
```

A computer cannot multiply, compare, or train directly on these words.

So we convert the sentence into a numerical form:

```text
"I drink tea"
↓
["I", "drink", "tea"]
↓
[1, 2, 3]
```

The numbers are not meaning yet. They are just lookup IDs.

In [27]:
sentence = "I drink tea"

print(sentence)
print(type(sentence))

I drink tea
<class 'str'>


# Part 2: Tokenization

Tokenization means breaking text into smaller pieces called tokens.

A token can be:

- a word
- a character
- a sub-word chunk
- punctuation
- a special marker like `[PAD]` or `[UNK]`

The simplest form is word-level tokenization.

In [28]:
sentence = "I drink tea"

tokens = sentence.split()

print(tokens)

['I', 'drink', 'tea']


## Practice 1: Tokenize Your Own Sentence

Write your own short sentence and split it into word tokens.

In [29]:
# Practice 1

my_sentence = "Red running shoes with white laces"

my_tokens = my_sentence.split()

print(my_tokens)

['Red', 'running', 'shoes', 'with', 'white', 'laces']


# Part 3: Vocabulary

A vocabulary is a master lookup table.

It assigns each known token a unique numerical ID.

Example:

```python
{
    "[PAD]": 0,
    "[UNK]": 1,
    "I": 2,
    "drink": 3,
    "tea": 4
}
```

Using this vocabulary:

```text
"I drink tea" → [2, 3, 4]
```

In [30]:
vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "I": 2,
    "drink": 3,
    "tea": 4
}

sentence = "I drink tea"
tokens = sentence.split()

ids = [vocab[token] for token in tokens]

print("Tokens:", tokens)
print("Token IDs:", ids)

Tokens: ['I', 'drink', 'tea']
Token IDs: [2, 3, 4]


# Part 4: Building a Vocabulary from Text

Now let's build a vocabulary automatically from a list of short item descriptions.

We will include two special tokens first:

```text
[PAD] = 0
[UNK] = 1
```

Then every unique word gets a new ID.

In [31]:
descriptions = [
    "red running shoes",
    "white running socks",
    "blue hiking shoes",
    "black leather boots",
    "red cotton shirt"
]

special_tokens = ["[PAD]", "[UNK]"]

vocab = {}

# Add special tokens first
for token in special_tokens:
    vocab[token] = len(vocab)

# Add words from descriptions
for description in descriptions:
    words = description.lower().split()

    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print(vocab)

{'[PAD]': 0, '[UNK]': 1, 'red': 2, 'running': 3, 'shoes': 4, 'white': 5, 'socks': 6, 'blue': 7, 'hiking': 8, 'black': 9, 'leather': 10, 'boots': 11, 'cotton': 12, 'shirt': 13}


## Practice 2: Build Your Own Vocabulary

Create 5 item descriptions for a search assistant.

Examples:

```text
"wireless bluetooth headphones"
"black gaming mouse"
"silver laptop stand"
```

Then build a vocabulary from them.

In [32]:
# Practice 2

my_descriptions = [
    "wireless bluetooth headphones",
    "black gaming mouse",
    "silver laptop stand",
    "waterproof hiking jacket",
    "cotton summer shirt"
]

my_vocab = {}

for token in ["[PAD]", "[UNK]"]:
    my_vocab[token] = len(my_vocab)

for description in my_descriptions:
    words = description.lower().split()

    for word in words:
        if word not in my_vocab:
            my_vocab[word] = len(my_vocab)

print(my_vocab)

{'[PAD]': 0, '[UNK]': 1, 'wireless': 2, 'bluetooth': 3, 'headphones': 4, 'black': 5, 'gaming': 6, 'mouse': 7, 'silver': 8, 'laptop': 9, 'stand': 10, 'waterproof': 11, 'hiking': 12, 'jacket': 13, 'cotton': 14, 'summer': 15, 'shirt': 16}


# Part 5: Converting Text to Token IDs

Once we have a vocabulary, we can convert text into a list of numbers.

This is sometimes called numericalization.

Example:

```text
"red running shoes"
↓
["red", "running", "shoes"]
↓
[2, 3, 4]
```

In [33]:
def text_to_ids(text, vocab):
    tokens = text.lower().split()
    ids = [vocab[token] for token in tokens]
    return tokens, ids

text = "red running shoes"

tokens, ids = text_to_ids(text, vocab)

print("Text:", text)
print("Tokens:", tokens)
print("IDs:", ids)

Text: red running shoes
Tokens: ['red', 'running', 'shoes']
IDs: [2, 3, 4]


# Part 6: Unknown Words and `[UNK]`

What happens if we see a word that is not in the vocabulary?

Example:

```text
"red running hat"
```

If `hat` is not in the vocabulary, we replace it with `[UNK]`.

The safer way to look up IDs is:

```python
vocab.get(token, vocab["[UNK]"])
```

This means:

```text
Use the token's ID if it exists.
Otherwise use the [UNK] ID.
```

In [34]:
def text_to_ids_safe(text, vocab):
    tokens = text.lower().split()
    ids = [vocab.get(token, vocab["[UNK]"]) for token in tokens]
    return tokens, ids

new_text = "red running hat"

tokens, ids = text_to_ids_safe(new_text, vocab)

print("Text:", new_text)
print("Tokens:", tokens)
print("IDs:", ids)

Text: red running hat
Tokens: ['red', 'running', 'hat']
IDs: [2, 3, 1]


## Practice 3: Test Unknown Words

Use your own vocabulary from Practice 2.

Write a new description with at least two words that are not in your vocabulary.

Convert the description into token IDs using `[UNK]`.

In [35]:
# Practice 3

new_description = "wireless red backpack"

tokens, ids = text_to_ids_safe(new_description, my_vocab)

print("Vocabulary:", my_vocab)
print("New description:", new_description)
print("Tokens:", tokens)
print("IDs:", ids)

Vocabulary: {'[PAD]': 0, '[UNK]': 1, 'wireless': 2, 'bluetooth': 3, 'headphones': 4, 'black': 5, 'gaming': 6, 'mouse': 7, 'silver': 8, 'laptop': 9, 'stand': 10, 'waterproof': 11, 'hiking': 12, 'jacket': 13, 'cotton': 14, 'summer': 15, 'shirt': 16}
New description: wireless red backpack
Tokens: ['wireless', 'red', 'backpack']
IDs: [2, 1, 1]


# Part 7: Padding with `[PAD]`

Neural networks often process many examples at the same time.

This is called a batch.

For matrix math to work, all sequences in a batch usually need the same length.

Example:

```text
"red shoes"              → [2, 4]
"white running socks"    → [5, 3, 6]
```

These have different lengths.

We can pad the shorter one:

```text
"red shoes"              → [2, 4, 0]
"white running socks"    → [5, 3, 6]
```

Here, `0` is the `[PAD]` token ID.

In [36]:
def pad_sequence(ids, max_length, pad_id=0):
    padded = ids.copy()

    while len(padded) < max_length:
        padded.append(pad_id)

    return padded[:max_length]

ids = [2, 4]
padded_ids = pad_sequence(ids, max_length=5, pad_id=vocab["[PAD]"])

print("Original IDs:", ids)
print("Padded IDs:", padded_ids)

Original IDs: [2, 4]
Padded IDs: [2, 4, 0, 0, 0]


# Part 8: Padding a Batch

Let's convert multiple descriptions into token IDs and pad them all to the same length.

In [37]:
batch_texts = [
    "red running shoes",
    "white socks",
    "blue hiking shoes",
    "black leather boots"
]

# Convert each text to IDs
batch_ids = []

for text in batch_texts:
    tokens, ids = text_to_ids_safe(text, vocab)
    batch_ids.append(ids)

print("Before padding:")
for text, ids in zip(batch_texts, batch_ids):
    print(text, "→", ids)

# Find maximum sequence length
max_length = max(len(ids) for ids in batch_ids)

# Pad each sequence
padded_batch = [
    pad_sequence(ids, max_length=max_length, pad_id=vocab["[PAD]"])
    for ids in batch_ids
]

print("\nAfter padding:")
for text, ids in zip(batch_texts, padded_batch):
    print(text, "→", ids)

Before padding:
red running shoes → [2, 3, 4]
white socks → [5, 6]
blue hiking shoes → [7, 8, 4]
black leather boots → [9, 10, 11]

After padding:
red running shoes → [2, 3, 4]
white socks → [5, 6, 0]
blue hiking shoes → [7, 8, 4]
black leather boots → [9, 10, 11]


## Practice 4: Pad Your Own Descriptions

Use your descriptions from Practice 2.

Convert them into IDs and pad them to the same length.

In [38]:
# Practice 4

my_batch_ids = []

for text in my_descriptions:
    tokens, ids = text_to_ids_safe(text, my_vocab)
    my_batch_ids.append(ids)

my_max_length = max(len(ids) for ids in my_batch_ids)

my_padded_batch = [
    pad_sequence(ids, max_length=my_max_length, pad_id=my_vocab["[PAD]"])
    for ids in my_batch_ids
]

for text, ids in zip(my_descriptions, my_padded_batch):
    print(text, "→", ids)

wireless bluetooth headphones → [2, 3, 4]
black gaming mouse → [5, 6, 7]
silver laptop stand → [8, 9, 10]
waterproof hiking jacket → [11, 12, 13]
cotton summer shirt → [14, 15, 16]


# Part 9: Word-Level Tokenization

Word-level tokenization splits text into words.

Example:

```text
"The unhappiest runner is sprinting"
↓
["The", "unhappiest", "runner", "is", "sprinting"]
```

## Pros

- Easy to understand
- Words often carry meaning
- Shorter than character-level tokenization

## Cons

- Vocabulary can become huge
- Rare words become a problem
- New words may become `[UNK]`

In [39]:
text = "The unhappiest runner is sprinting"

word_tokens = text.split()

print(word_tokens)
print("Number of word-level tokens:", len(word_tokens))

['The', 'unhappiest', 'runner', 'is', 'sprinting']
Number of word-level tokens: 5


# Part 10: Character-Level Tokenization

Character-level tokenization splits text into individual characters.

Example:

```text
"tea"
↓
["t", "e", "a"]
```

## Pros

- Very small vocabulary
- Can handle almost any new word

## Cons

- Sequences become very long
- Each token has very little meaning on its own

In [40]:
text = "The unhappiest runner is sprinting"

character_tokens = list(text)

print(character_tokens)
print("Number of character-level tokens:", len(character_tokens))

['T', 'h', 'e', ' ', 'u', 'n', 'h', 'a', 'p', 'p', 'i', 'e', 's', 't', ' ', 'r', 'u', 'n', 'n', 'e', 'r', ' ', 'i', 's', ' ', 's', 'p', 'r', 'i', 'n', 't', 'i', 'n', 'g']
Number of character-level tokens: 34


# Part 11: Sub-Word Tokenization

Sub-word tokenization breaks words into smaller meaningful chunks.

Example:

```text
"walking" → ["walk", "ing"]
"unhappiest" → ["un", "happi", "est"]
"sprinting" → ["sprint", "ing"]
```

This is the strategy commonly used in professional language models.

It is a balance between word-level and character-level tokenization.

In [41]:
# A simple manual sub-word breakdown
text = "The unhappiest runner is sprinting"

subword_tokens = ["The", "un", "happi", "est", "runner", "is", "sprint", "ing"]

print(subword_tokens)
print("Number of sub-word tokens:", len(subword_tokens))

['The', 'un', 'happi', 'est', 'runner', 'is', 'sprint', 'ing']
Number of sub-word tokens: 8


## Practice 5: Compare Tokenization Methods

Use the sentence:

```text
The unhappiest runner is sprinting
```

Break it down using:

1. Word-level tokenization
2. Character-level tokenization
3. Logical sub-word chunks

Then compare:

- Which has the fewest tokens?
- Which has the most tokens?
- Which gives the model the most useful chunks of meaning?

In [42]:
# Practice 5

sentence = "The unhappiest runner is sprinting"

word_level = sentence.split()
character_level = list(sentence)
subword_level = ["The", "un", "happi", "est", "runner", "is", "sprint", "ing"]

print("Word-level:", word_level)
print("Count:", len(word_level))

print("\nCharacter-level:", character_level)
print("Count:", len(character_level))

print("\nSub-word level:", subword_level)
print("Count:", len(subword_level))

Word-level: ['The', 'unhappiest', 'runner', 'is', 'sprinting']
Count: 5

Character-level: ['T', 'h', 'e', ' ', 'u', 'n', 'h', 'a', 'p', 'p', 'i', 'e', 's', 't', ' ', 'r', 'u', 'n', 'n', 'e', 'r', ' ', 'i', 's', ' ', 's', 'p', 'r', 'i', 'n', 't', 'i', 'n', 'g']
Count: 34

Sub-word level: ['The', 'un', 'happi', 'est', 'runner', 'is', 'sprint', 'ing']
Count: 8


# Part 12: Token IDs Are Not Meaning

Token IDs are labels, not meaning.

Example:

```text
cat → 45
kitten → 9002
```

This does not mean kitten is mathematically greater than cat.

It only means:

```text
cat is stored at ID 45
kitten is stored at ID 9002
```

The model learns meaning later using embeddings.

Embeddings are the next step after token IDs.

In [43]:
example_vocab = {
    "cat": 45,
    "kitten": 9002
}

print(example_vocab)

print("Is kitten mathematically 200x more meaningful than cat?")
print("No. These are just lookup IDs.")

{'cat': 45, 'kitten': 9002}
Is kitten mathematically 200x more meaningful than cat?
No. These are just lookup IDs.


# Part 13: Context Window

A context window is the maximum number of tokens a model can process at one time.

Example:

```text
Model limit: 8,000 tokens
Input text: 9,000 tokens
Problem: 1,000 tokens do not fit
```

This matters because long prompts, documents, and conversations can exceed the model's context limit.

In [44]:
def estimate_tokens_from_words(word_count):
    # Rule of thumb:
    # 1000 tokens is approximately 750 English words
    return int(word_count * (1000 / 750))

word_count = 750
estimated_tokens = estimate_tokens_from_words(word_count)

print(f"{word_count} words is roughly {estimated_tokens} tokens.")

750 words is roughly 1000 tokens.


## Practice 6: Estimate Token Counts

Use the rule of thumb:

```text
1,000 tokens ≈ 750 words
```

Estimate token counts for:

1. 100 words
2. 500 words
3. 1,200 words
4. 2,000 words

In [45]:
# Practice 6

word_counts = [100, 500, 1200, 2000]

for count in word_counts:
    estimated = estimate_tokens_from_words(count)
    print(f"{count} words ≈ {estimated} tokens")

100 words ≈ 133 tokens
500 words ≈ 666 tokens
1200 words ≈ 1600 tokens
2000 words ≈ 2666 tokens


# Part 14: Prompt Compression Practice

If tokens are limited, we may need to rewrite prompts to be shorter.

Long prompt:

```text
Please carefully analyze the following long product description and tell me whether it would be suitable for a customer who is specifically looking for a lightweight waterproof hiking jacket for summer travel.
```

Shorter prompt:

```text
Is this product a lightweight waterproof summer hiking jacket?
```

Both ask for similar information, but the shorter prompt uses fewer tokens.

In [46]:
long_prompt = "Please carefully analyze the following long product description and tell me whether it would be suitable for a customer who is specifically looking for a lightweight waterproof hiking jacket for summer travel."
short_prompt = "Is this product a lightweight waterproof summer hiking jacket?"

print("Long prompt word count:", len(long_prompt.split()))
print("Short prompt word count:", len(short_prompt.split()))

print("Estimated long prompt tokens:", estimate_tokens_from_words(len(long_prompt.split())))
print("Estimated short prompt tokens:", estimate_tokens_from_words(len(short_prompt.split())))

Long prompt word count: 32
Short prompt word count: 9
Estimated long prompt tokens: 42
Estimated short prompt tokens: 12


## Practice 7: Rewrite a Prompt

Rewrite this wordy prompt in fewer words:

```text
I would like you to examine this item description and determine whether or not it is a strong match for someone who wants comfortable shoes for running outdoors during rainy weather.
```

Goal: Keep the same meaning with fewer words.

In [47]:
# Practice 7

wordy_prompt = "I would like you to examine this item description and determine whether or not it is a strong match for someone who wants comfortable shoes for running outdoors during rainy weather."

shorter_prompt = "Is this item good for comfortable rainy outdoor running shoes?"

print("Wordy prompt:", wordy_prompt)
print("Word count:", len(wordy_prompt.split()))

print("\nShorter prompt:", shorter_prompt)
print("Word count:", len(shorter_prompt.split()))

Wordy prompt: I would like you to examine this item description and determine whether or not it is a strong match for someone who wants comfortable shoes for running outdoors during rainy weather.
Word count: 31

Shorter prompt: Is this item good for comfortable rainy outdoor running shoes?
Word count: 10


# Part 15: Optional Token Counting with `tiktoken`

Professional AI systems use tokenizers.

For OpenAI models, a common tokenizer library is `tiktoken`.

If you have it installed, you can run:

```bash
pip install tiktoken
```

Then use it to count tokens.

If it is not installed, this notebook will skip this section safely.

In [48]:
try:
    import tiktoken

    encoding = tiktoken.get_encoding("cl100k_base")

    text = "The unhappiest runner is sprinting."
    tokens = encoding.encode(text)

    print("Text:", text)
    print("Token IDs:", tokens)
    print("Token count:", len(tokens))

except ImportError:
    print("tiktoken is not installed.")
    print("To install it, run: pip install tiktoken")

Text: The unhappiest runner is sprinting.
Token IDs: [791, 31175, 680, 13744, 23055, 374, 38949, 287, 13]
Token count: 9


# Part 16: API Concept

An API, or Application Programming Interface, lets one program talk to another program.

Analogy:

| Concept | Analogy |
|---|---|
| Your Streamlit app | Customer |
| API | Waiter |
| AI model | Chef |
| Response | Food returned to customer |

The app sends a request through the API.  
The model processes it.  
The API returns the response.

# Part 17: API Key Safety

An API key is a sensitive credential.

Treat it like:

```text
password
credit card
private key
```

Do not hardcode it into a Python file.

Do not upload it to GitHub.

Safer Streamlit structure:

```text
project_folder/
    app.py
    .streamlit/
        secrets.toml
```

Inside `secrets.toml`:

```toml
OPENAI_API_KEY = "your-api-key-here"
```

Inside Streamlit:

```python
import streamlit as st

api_key = st.secrets["OPENAI_API_KEY"]
```

In [49]:
# This is a safe demonstration.
# Do not paste a real API key into this notebook.

fake_api_key = "do-not-use-real-keys-in-notebooks"

print("Never hardcode real API keys.")
print("Use secrets.toml or environment variables instead.")

Never hardcode real API keys.
Use secrets.toml or environment variables instead.


# Part 18: Independent Work Activity 1  
## Custom Vocabulary and Token Mapping

Goal: Understand how to manually convert text into machine-readable numerical format.

Tasks:

1. Select five short item descriptions.
2. Create a vocabulary with every unique word.
3. Include `[PAD]` and `[UNK]`.
4. Convert a new 10-word description into token IDs.
5. Replace unknown words with `[UNK]`.
6. Pad all vectors to the same length.

In [50]:
# Independent Work Activity 1 Template

item_descriptions = [
    "red running shoes with white laces",
    "blue waterproof hiking jacket",
    "black wireless gaming mouse",
    "silver laptop stand adjustable height",
    "cotton summer shirt short sleeves"
]

# Step 1: Build vocabulary
activity_vocab = {"[PAD]": 0, "[UNK]": 1}

for description in item_descriptions:
    for word in description.lower().split():
        if word not in activity_vocab:
            activity_vocab[word] = len(activity_vocab)

print("Vocabulary:")
print(activity_vocab)

# Step 2: Convert new description
new_description = "red lightweight running hat for rainy outdoor training"

new_tokens, new_ids = text_to_ids_safe(new_description, activity_vocab)

print("\nNew description:", new_description)
print("Tokens:", new_tokens)
print("IDs:", new_ids)

# Step 3: Pad to target length
target_length = 15
new_ids_padded = pad_sequence(new_ids, max_length=target_length, pad_id=activity_vocab["[PAD]"])

print("Padded IDs:", new_ids_padded)
print("Length:", len(new_ids_padded))

Vocabulary:
{'[PAD]': 0, '[UNK]': 1, 'red': 2, 'running': 3, 'shoes': 4, 'with': 5, 'white': 6, 'laces': 7, 'blue': 8, 'waterproof': 9, 'hiking': 10, 'jacket': 11, 'black': 12, 'wireless': 13, 'gaming': 14, 'mouse': 15, 'silver': 16, 'laptop': 17, 'stand': 18, 'adjustable': 19, 'height': 20, 'cotton': 21, 'summer': 22, 'shirt': 23, 'short': 24, 'sleeves': 25}

New description: red lightweight running hat for rainy outdoor training
Tokens: ['red', 'lightweight', 'running', 'hat', 'for', 'rainy', 'outdoor', 'training']
IDs: [2, 1, 3, 1, 1, 1, 1, 1]
Padded IDs: [2, 1, 3, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
Length: 15


# Part 19: Independent Work Activity 2  
## Token Efficiency and Sub-Word Analysis

Goal: Analyze how tokenization strategies affect context window usage.

Tasks:

1. Write a sentence with complex or compound words.
2. Break it into word-level tokens.
3. Break it into character-level tokens.
4. Break it into logical sub-word chunks.
5. Compare token counts.
6. Rewrite a wordy prompt to use fewer tokens.

In [51]:
# Independent Work Activity 2 Template

complex_sentence = "The unhappiest ultramarathon runner is sprinting extraordinarily fast"

word_tokens = complex_sentence.split()
character_tokens = list(complex_sentence)
subword_tokens = [
    "The", "un", "happi", "est",
    "ultra", "marathon",
    "runner", "is",
    "sprint", "ing",
    "extra", "ordinary", "ly",
    "fast"
]

print("Sentence:", complex_sentence)

print("\nWord-level tokens:")
print(word_tokens)
print("Count:", len(word_tokens))

print("\nCharacter-level tokens:")
print(character_tokens)
print("Count:", len(character_tokens))

print("\nSub-word tokens:")
print(subword_tokens)
print("Count:", len(subword_tokens))

Sentence: The unhappiest ultramarathon runner is sprinting extraordinarily fast

Word-level tokens:
['The', 'unhappiest', 'ultramarathon', 'runner', 'is', 'sprinting', 'extraordinarily', 'fast']
Count: 8

Character-level tokens:
['T', 'h', 'e', ' ', 'u', 'n', 'h', 'a', 'p', 'p', 'i', 'e', 's', 't', ' ', 'u', 'l', 't', 'r', 'a', 'm', 'a', 'r', 'a', 't', 'h', 'o', 'n', ' ', 'r', 'u', 'n', 'n', 'e', 'r', ' ', 'i', 's', ' ', 's', 'p', 'r', 'i', 'n', 't', 'i', 'n', 'g', ' ', 'e', 'x', 't', 'r', 'a', 'o', 'r', 'd', 'i', 'n', 'a', 'r', 'i', 'l', 'y', ' ', 'f', 'a', 's', 't']
Count: 69

Sub-word tokens:
['The', 'un', 'happi', 'est', 'ultra', 'marathon', 'runner', 'is', 'sprint', 'ing', 'extra', 'ordinary', 'ly', 'fast']
Count: 14


# Final Reflection

Answer these questions:

1. Why can’t computers process raw text directly?
2. What is a vocabulary?
3. What is tokenization?
4. What is the difference between word-level, character-level, and sub-word tokenization?
5. What problem does `[UNK]` solve?
6. What problem does `[PAD]` solve?
7. Why are token IDs not the same as meaning?
8. What is a context window?
9. Why does token efficiency matter?
10. Why should API keys never be hardcoded?

In [52]:
# Final Reflection Answers

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.
# 9.
# 10.